# 03 – Crop Generation for Classification Stage

The second stage of the pipeline classifies each detected trash object into
a material category.  This notebook extracts bounding-box crops from the
labelled detection dataset and builds a manifest CSV that drives the
classification training workflow.

**Output:**
- Crops in `datasets/processed/classification/crops/<class>/`
- Manifest at `datasets/processed/classification/crops_manifest.csv`

## 1 · Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("__file__").resolve().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from trash_detection.crops import save_crops_from_yolo_labels, build_crops_manifest
from trash_detection.viz import show_image_grid
import cv2

# ── paths ────────────────────────────────────────────────────────────────────
DETECTION_ROOT = REPO_ROOT / "datasets" / "detection"
CROPS_ROOT     = REPO_ROOT / "datasets" / "processed" / "classification"
MANIFEST_PATH  = CROPS_ROOT / "crops_manifest.csv"

# We generate crops from the *train* split (ground-truth labels exist there)
IMAGES_DIR = DETECTION_ROOT / "images" / "train"
LABELS_DIR = DETECTION_ROOT / "labels" / "train"

print("Images dir:", IMAGES_DIR)
print("Labels dir:", LABELS_DIR)
print("Crops root:", CROPS_ROOT)

## 2 · Generate Crops from Ground Truth

Each bounding box in the YOLO labels is cropped from its source image,
padded by 10 % on each side, and resized to 224 × 224 pixels.

At this stage all crops are saved under a single `trash/` sub-folder because
material labels are not yet assigned (see Section 4 for relabelling guidance).

In [ ]:
if IMAGES_DIR.exists() and LABELS_DIR.exists():
    records = save_crops_from_yolo_labels(
        images_dir=IMAGES_DIR,
        labels_dir=LABELS_DIR,
        output_dir=CROPS_ROOT / "crops",
        class_name="trash",   # placeholder – will be re-organised manually
        padding=0.10,
        target_size=(224, 224),
    )
    print(f"Generated {len(records)} crops.")
else:
    print("Detection dataset not found. Run notebook 02 and label your data first.")
    records = []

## 3 · Build Metadata CSV

In [ ]:
if records:
    manifest_path = build_crops_manifest(records, MANIFEST_PATH)
    print(f"Manifest saved → {manifest_path}")

    import pandas as pd
    df = pd.read_csv(manifest_path)
    display(df.head(10))
    print(f"\nShape: {df.shape}")
else:
    print("No records to save.")

## 4 · Manual Relabelling Guidance

After crop generation, move (or copy) each crop into the correct material
sub-folder to build the classification dataset:

```
datasets/processed/classification/crops/
├── plastic/    ← PET bottles, bags, wrappers
├── glass/      ← bottles, jars, broken glass
├── metal/      ← cans, foil, wire
├── paper/      ← cardboard, newspapers, cups
└── other/      ← cigarettes, organic waste, unknown
```

### Tips

- Use a file manager with thumbnail view for speed.
- When in doubt, assign to `other`.
- Aim for at least **200 samples per class** for reliable training.
- Record any tricky edge cases in `datasets/raw/metadata/DATA_SOURCES.md`.

### Shell one-liner to create folders

```bash
mkdir -p datasets/processed/classification/crops/{plastic,glass,metal,paper,other}
```

## 5 · Verify Crops

In [ ]:
# Show a sample grid of the generated crops
from trash_detection.io import find_images

CLASSES = ["plastic", "glass", "metal", "paper", "other", "trash"]
sample_paths = []
sample_titles = []

for cls in CLASSES:
    cls_dir = CROPS_ROOT / "crops" / cls
    if cls_dir.exists():
        imgs = find_images(cls_dir)[:3]
        sample_paths.extend(imgs)
        sample_titles.extend([cls] * len(imgs))

if sample_paths:
    images = [cv2.imread(str(p)) for p in sample_paths]
    images = [img for img in images if img is not None]
    show_image_grid(images[:16], titles=sample_titles[:16], cols=4)
    print(f"Showing {len(images[:16])} sample crops.")
else:
    print("No crops found yet – run Section 2 first.")